In [6]:
# =========================================================
# STEP 1: HARD FORCED REGISTRY PATH INJECTION
# =========================================================
import sys
import os
import pandas as pd
import numpy as np

# Define your absolute root explicitly to eliminate directory layer guessing games
PROJECT_ROOT = "/Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting"

# Reset working directory back to base root state
os.chdir(PROJECT_ROOT)

# Inject path directly to the front of Python's memory registry
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Direct data mapping paths
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"🎯 Forced Working Directory: {os.getcwd()}")
print(f"📁 Checking for 'src' folder: {os.path.exists(os.path.join(PROJECT_ROOT, 'src'))}")

# =========================================================
# STEP 2: FEATURE ENGINEERING MATRIX PIPELINE
# =========================================================
from src.features import build_feature_matrix

print("Loading raw files via absolute machine mapping...")
eua = pd.read_csv(os.path.join(RAW_DIR, "raw_eua.csv"), index_col=0, parse_dates=True)
ttf = pd.read_csv(os.path.join(RAW_DIR, "raw_ttf.csv"), index_col=0, parse_dates=True)
coal = pd.read_csv(os.path.join(RAW_DIR, "raw_coal.csv"), index_col=0, parse_dates=True)
weather = pd.read_csv(os.path.join(RAW_DIR, "raw_weather.csv"), index_col=0, parse_dates=True)
generation = pd.read_csv(os.path.join(RAW_DIR, "raw_generation.csv"), index_col=0, parse_dates=True)
policy_events = pd.read_csv(os.path.join(PROCESSED_DIR, "policy_events.csv"), index_col=0, parse_dates=True)

print("⚡ Running feature engineering transformations...")
# FIXED: Swapped 'generation' and 'weather' positions to match function signature order
final_matrix = build_feature_matrix(eua, ttf, coal, generation, weather, policy_events)

# Save the final matrix out to your processed data layer
output_path = os.path.join(PROCESSED_DIR, "final_feature_matrix.csv")
final_matrix.to_csv(output_path)
print(f"🎉 SUCCESS! Final feature matrix built and stored at: {output_path}")

🎯 Forced Working Directory: /Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting
📁 Checking for 'src' folder: True
Loading raw files via absolute machine mapping...
⚡ Running feature engineering transformations...
🎉 SUCCESS! Final feature matrix built and stored at: /Users/etiennerv/Jts/EUA Forecasting/EUA-forecasting/data/processed/final_feature_matrix.csv


In [7]:
from statsmodels.tsa.stattools import adfuller

def adf_report(series, name):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"--- ADF Test: {name} ---")
    print(f"  p-value: {result[1]:.4f}")
    if result[1] < 0.05:
        print("  → STATIONARY (Good for linear modeling)\n")
    else:
        print("  → NON-STATIONARY (Needs differencing)\n")

df = pd.read_csv("data/processed/final_feature_matrix.csv", index_col=0, parse_dates=True)

# Test price levels vs log returns
adf_report(df["eua_price"], "EUA Price (Levels)")
adf_report(df["d_log_eua"], "Δ Log EUA (Returns)")

--- ADF Test: EUA Price (Levels) ---
  p-value: 0.0109
  → STATIONARY (Good for linear modeling)

--- ADF Test: Δ Log EUA (Returns) ---
  p-value: 0.0000
  → STATIONARY (Good for linear modeling)

